In [1]:
# 사용자 정보를 기반으로 영화를 추천하는 서비스
# 1번 사용자한테 추천할 영화 5개를 선택
# 1번 사용자가 2번 영화에 별점을 몇점 줄것인지 예측

In [2]:
# ratings_small.csv에 있는 데이터를 가져와서 DataFrame으로 변환
import pandas as pd

df_ratings = pd.read_csv('../data/csv/ratings_small.csv')

# tmdb_5000_credits.csv에 있는 데이터를 가져와서 DataFrame로 변환
df_movies = pd.read_csv('../data/csv/tmdb_5000_credits.csv')


In [3]:
# df_movies에 있는 영화제목을 df_ratings에 추가
df_movies.columns = ['movieId', 'title', 'cast', 'crew']
df = df_ratings.merge(df_movies[['movieId', 'title']], on='movieId')

In [4]:
# userId의 값을 0부터 맵핑
# 1. 중복된 userId를 제거
user_ids = df['userId'].unique().tolist()
# 2. [{1 : 0},{2 : 1}]형태로 변환 {userId : 순서}
user_to_index = {x : i for i,x in enumerate(user_ids)}


In [5]:
# movieId의 값을 0부터 맵핑
movie_ids = df['movieId'].unique().tolist()
# 2. [{1 : 0},{2 : 1}]형태로 변환 {userId : 순서}
movie_to_index = {x : i for i,x in enumerate(movie_ids)}

In [6]:
# df에 movie_ids와 user_ids를 추가
df['user'] = df['userId'].map(user_to_index)
df['movie'] = df['movieId'].map(movie_to_index)

In [7]:
# 사용자 수, 아이템(영화)수를 계산
num_users = len(user_ids)
num_movies = len(movie_ids)

In [8]:
import tensorflow as tf
# 학습에 사용할 데이터 셋을 준비
# 입력은 딕셔너리, 정답은 리스트 형태로
user_data = df['user'].values.astype('int32')
movie_data = df['movie'].values.astype('int32')
ratings = df['rating'].values.astype('float32')
# from_tensor_slices((X, y))
dataset = tf.data.Dataset.from_tensor_slices(({
	'user_in' : user_data,
	'movie_in' : movie_data
	}, ratings))
train_ds = dataset.cache().shuffle(100).batch(32).prefetch(32)

In [9]:
# 모델 설계 : Sequential 사용 못함
from tensorflow.keras import layers, models

# 여러개의 입력 중 사용자 점보와 영화 정보를 어디서 가져와야 하는지를 지정
user_input = layers.Input(shape=(1,), name='user_in')
movie_input = layers.Input(shape=(1,), name='movie_in')

# 임베딩
user_emb = layers.Embedding(input_dim=num_users, output_dim=16, name='user_emb')(user_input)
movie_emb = layers.Embedding(input_dim=num_movies, output_dim=16, name='movie_emb')(movie_input)

# 벡터 평탄화
user_vec = layers.Flatten()(user_emb)
movie_vec = layers.Flatten()(movie_emb)

# 두 벡터를 결합
x = layers.Concatenate()([user_vec, movie_vec])

# 은닉층 추가
x = layers.Dense(32, activation='relu')(x)
x = layers.Dense(16, activation='relu')(x)

# 출력층
outputs = layers.Dense(1, activation='linear')(x)

model = models.Model(inputs=[user_input, movie_input], outputs=outputs)

# 모델 설정
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

In [10]:
history = model.fit(train_ds, epochs=10, verbose=1)

Epoch 1/10
581/581 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 2.2793 - mae: 1.1023
Epoch 2/10
581/581 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.8130 - mae: 0.6999
Epoch 3/10
581/581 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.7563 - mae: 0.6728
Epoch 4/10
581/581 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.7401 - mae: 0.6658
Epoch 5/10
581/581 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.7260 - mae: 0.6597
Epoch 6/10
581/581 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.7086 - mae: 0.6497
Epoch 7/10
581/581 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.6889 - mae: 0.6384
Epoch 8/10
581/581 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.6745 - mae: 0.6318
Epoch 9/10
581/581 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.6563 - mae: 0.6209
Epoch 10/10
581/581 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.6361 - mae: 0.6115


In [11]:
import numpy as np
test_user_idx = 0 # 실제 1번 사용자
test_movie_idx = 2 # 2번 영화  '2001: A Space Odyssey'
# df[df['movie'] == 2]['title'].values[0]

test_user = np.array([test_user_idx])
test_movie = np.array([test_movie_idx])

prediction = model.predict({
	'user_in' : test_user, 'movie_in' : test_movie},
	verbose=1)

print('--------------------')
res = prediction[0][0]

res = np.round(res/0.5)*0.5 # 예상 평점이 2.8이면 반올림해서 3 
def get_movie_title(movie_to_index_id):
	for idx in movie_to_index:
		if movie_to_index_id == movie_to_index[idx]:
			return df_movies[df_movies['movieId'] == idx]['title'].values[0]
	return "없음"
title = get_movie_title(test_movie_idx)
print(f'사용자 {test_user_idx}의 영화 {test_movie_idx}에 대한 예상 평점 : {res:.2f}점')
df[df['user'] == 0][['movie', 'title', 'rating']]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 192ms/step
--------------------
사용자 0의 영화 2에 대한 예상 평점 : 3.50점


,movie,title,rating
0,0,American Pie,4.0
1,1,Jay and Silent Bob Strike Back,2.0


In [12]:
import numpy as np
test_user_id = 1 # 실제 1번 회원
test_movie_id = 62 # '2001: A Space Odyssey'
# df[df['movie'] == 2]['title'].values[0]

test_user = np.array([user_to_index[test_user_id]])
test_movie = np.array([movie_to_index[test_movie_id]])

prediction = model.predict({
	'user_in' : test_user, 'movie_in' : test_movie},
	verbose=1)

print('-------------')
res = prediction[0][0]
res = np.round(res / 0.5) * 0.5 # 예상 평점이 2.8이면 반올림해서 3

def get_movie_title(movie_id):
	return df_movies[df_movies['movieId'] == movie_id]['title'].values[0]
	

title = get_movie_title(test_movie_id)
print(f'사용자 {test_user_id}의 영화 "{title}"에 대한 예상 평점 : {res:.2f}점')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
-------------
사용자 1의 영화 "2001: A Space Odyssey"에 대한 예상 평점 : 3.50점


In [18]:
test_user_id = 1

# 1번 사용자에게 추천하는 영화 목록
# 1번 사용자가 본 영화를 제외하는 작업

# 1번 사용자가 본 영화 id들
watched_movie_idxs = df[df['userId'] == test_user_id]['movie'].unique()
# 전체 영화 id들
all_movie_idxs = np.array(list(movie_to_index.values()))
# 1번 사용자가 안본 영화 id들
candidate_movie_idxs = all_movie_idxs[~np.isin(all_movie_idxs, watched_movie_idxs)]

# [test_user_id]*3 # 리스트 * 3은 리스트가 3번 반복
# np.array([test_user_id]) * 3 # 넘파이 리스트 * 3은 리스트의 값을 3배 => [3]

test_user_idxs = np.array([user_to_index[test_user_id]] * len(candidate_movie_idxs))

prediction = model.predict({
	'user_in' : test_user_idxs,
	'movie_in' : candidate_movie_idxs
	})

# 정렬. argsort() 값을 기준으로 오름차순으로 정렬하여 인덱스를 뒤에서 5개 반환
# [::-1]역순
top_idxs = prediction.flatten().argsort()[-5:][::-1]

# {영화번호 : 0, 영화번호 : 1} => {0 : 영화번호, 1 : 영화번호}. k와 v를 반대로
top_ids = {i:movie for movie,i in movie_to_index.items()}
print(top_ids)
top_titles=[get_movie_title(top_ids[i]) for i in top_idxs]
for i, idx in enumerate(top_idxs):
	print(f'{i+1}위 : {top_titles[i]}(예상 평점 : {prediction[idx][0]:.2f}점)')


27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
{0: 2105, 1: 2294, 2: 62, 3: 153, 4: 161, 5: 165, 6: 168, 7: 186, 8: 223, 9: 235, 10: 248, 11: 253, 12: 261, 13: 272, 14: 292, 15: 296, 16: 314, 17: 319, 18: 350, 19: 364, 20: 377, 21: 454, 22: 468, 23: 480, 24: 497, 25: 500, 26: 508, 27: 509, 28: 539, 29: 550, 30: 585, 31: 586, 32: 587, 33: 588, 34: 590, 35: 592, 36: 593, 37: 616, 38: 595, 39: 866, 40: 1271, 41: 1378, 42: 2841, 43: 2959, 44: 141, 45: 173, 46: 289, 47: 329, 48: 380, 49: 431, 50: 435, 51: 440, 52: 464, 53: 594, 54: 913, 55: 1073, 56: 1125, 57: 1213, 58: 1257, 59: 1259, 60: 1265, 61: 1282, 62: 1372, 63: 1544, 64: 1858, 65: 1954, 66: 1961, 67: 2018, 68: 2034, 69: 2054, 70: 2080, 71: 2085, 72: 2100, 73: 2110, 74: 2114, 75: 2140, 76: 2268, 77: 2289, 78: 2454, 79: 2770, 80: 3034, 81: 3040, 82: 3060, 83: 104, 84: 231, 85: 277, 86: 597, 87: 788, 88: 1777, 89: 1997, 90: 2023, 91: 2355, 92: 2502, 93: 4995, 94: 111, 95: 1250, 96: 1639, 97: 1687, 98: 1909, 99: 2001, 100: 8874, 101: 207, 102: 